# BrainScan-v2 — Étape 1 : Classification des tumeurs cérébrales (GoogLeNet)

## 1.2 — Préparation des données, split 70/15/15 et prétraitement

**Objectifs du notebook**

1. Reproduire l'inventaire du dataset (extensions autorisées + `try/except`).
2. **Diviser les images en trois ensembles** : Train (70 %), Validation (15 %), Test (15 %),
   de façon **stratifiée** (les proportions de classes sont conservées dans chaque ensemble).
3. **Compter les images** présentes dans chaque ensemble.
4. Écrire un **dataset nettoyé** (RGB, redimensionné) sur disque pour l'entraînement.
5. Charger les images par lots avec `tf.keras.utils.image_dataset_from_directory()`
   et `shuffle=True` pour l'ensemble d'entraînement.
6. Définir le **prétraitement** Keras (redimensionnement, mise à l'échelle, normalisation)
   et les **augmentations** (flip, flou, bruit) appliquées **uniquement à l'entraînement**.

In [ ]:
# Cellule 0 — Imports, configuration et chemins du projet
"""Cellules communes réutilisées par les 4 notebooks de l'étape 1 (classification)."""

import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib

if matplotlib.get_backend().lower() == "agg":  # exécution headless (Docker / nbconvert)
    try:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    except Exception:
        pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# --------------------------------------------------------------------------------------
# Configuration du projet
# --------------------------------------------------------------------------------------
def _detect_project_root() -> Path:
    """Racine du projet (compatible exécution locale et exécution dans Docker /app)."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "Data").is_dir() and (candidate / "src").is_dir():
            return candidate
    if Path("/app/data/Data").is_dir():
        return Path("/app")
    raise FileNotFoundError("Dataset introuvable : « data/Data » manquant depuis " + str(Path.cwd()))


PROJECT_ROOT = _detect_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed" / "classification"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for _d in (ARTIFACTS_DIR, MODELS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

#: Graine aléatoire globale (reproductibilité de bout en bout).
SEED = 42

#: Extensions d'images autorisées par le cahier des charges.
ALLOWED_EXTENSIONS = {".jpeg", ".jpg", ".bmp", ".png"}

#: Libellés lisibles pour l'affichage (correspondance dossier -> pathologie).
CLASS_DISPLAY = {
    "glioma": "Gliome",
    "meningioma": "Méningiome",
    "notumor": "Aucune tumeur",
    "pituitary": "Hypophyse",
}

#: Classes du dataset, déduites des noms de dossiers (ordre alphabétique).
class_names = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_DIR         : {RAW_DIR}")
print(f"PROCESSED_DIR   : {PROCESSED_DIR}")
print(f"ARTIFACTS_DIR   : {ARTIFACTS_DIR}")
print(f"MODELS_DIR      : {MODELS_DIR}")
print(f"Classes         : {class_names}")


### 1.10 — Inventaire sécurisé du dataset

Même procédure que le notebook 1.1 : filtrage par extension puis ouverture sous
`try/except`. Le manifeste du notebook précédent est réutilisé s'il existe (gain de temps),
sinon il est reconstruit.

In [ ]:
# Cellule 2.1 — Construction / rechargement du manifeste
from PIL import Image


def build_manifest() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Inventaire des images valides (extension autorisée + ouverture sans erreur)."""
    rows = []
    for class_dir in sorted(p for p in RAW_DIR.iterdir() if p.is_dir()):
        for f in sorted(class_dir.rglob("*")):
            if f.is_file() and f.suffix.lower() in ALLOWED_EXTENSIONS:
                rows.append({"label": class_dir.name, "file": f.name, "path": str(f)})

    valid, broken = [], []
    for rec in rows:
        try:
            with Image.open(rec["path"]) as im:
                im.verify()
            valid.append(rec)
        except Exception as exc:  # noqa: BLE001
            broken.append({**rec, "error": f"{type(exc).__name__}: {exc}"})

    return pd.DataFrame(valid), pd.DataFrame(broken)


MANIFEST_CSV = REPORTS_DIR / "classification_manifest.csv"

if MANIFEST_CSV.exists():
    manifest_df = pd.read_csv(MANIFEST_CSV)
    broken_df = pd.DataFrame(columns=["label", "file", "path", "error"])
    print(f"Manifeste réutilisé : {MANIFEST_CSV} ({len(manifest_df)} images)")
else:
    manifest_df, broken_df = build_manifest()
    manifest_df.to_csv(MANIFEST_CSV, index=False)
    print(f"Manifeste reconstruit : {len(manifest_df)} images valides, {len(broken_df)} en erreur")

class_names = sorted(manifest_df["label"].unique().tolist())
print(f"Classes : {class_names}")


### 1.11 — Split stratifié Train / Validation / Test (70 / 15 / 15)

Le découpage est **stratifié** : `sklearn.model_selection.train_test_split` avec
`stratify=label` garantit que chaque ensemble conserve la même proportion de chaque
classe. Le second appel scinde le reliquat 30 % en Validation (15 %) et Test (15 %).

La graine `SEED = 42` rend l'opération **reproductible**.

In [ ]:
# Cellule 2.2 — Découpage stratifié 70/15/15
from sklearn.model_selection import train_test_split

SPLIT_RATIOS = {"train": 0.70, "valid": 0.15, "test": 0.15}
assert abs(sum(SPLIT_RATIOS.values()) - 1.0) < 1e-9

train_df, _rest_df = train_test_split(
    manifest_df,
    test_size=1.0 - SPLIT_RATIOS["train"],
    stratify=manifest_df["label"],
    random_state=SEED,
)

valid_df, test_df = train_test_split(
    _rest_df,
    test_size=SPLIT_RATIOS["test"] / (SPLIT_RATIOS["valid"] + SPLIT_RATIOS["test"]),
    stratify=_rest_df["label"],
    random_state=SEED,
)

splits = {"train": train_df, "valid": valid_df, "test": test_df}

for name, df in splits.items():
    df.attrs["split"] = name

print("Répartition demandée :", SPLIT_RATIOS)
for name, df in splits.items():
    print(f"  {name:<6s} : {len(df):>5d} images ({len(df)/len(manifest_df):6.2%})")

# Contrôle de disjointude
paths_by_split = {name: set(df["path"]) for name, df in splits.items()}
assert paths_by_split["train"].isdisjoint(paths_by_split["valid"])
assert paths_by_split["train"].isdisjoint(paths_by_split["test"])
assert paths_by_split["valid"].isdisjoint(paths_by_split["test"])
print("\nContrôle de disjointude des 3 ensembles : OK")


### 1.12 — Comptage des images par ensemble

Vérification que les effectifs respectent le ratio 70/15/15 et que la **stratification**
a bien été conservée (proportions de classes quasi identiques dans les 3 ensembles).

In [ ]:
# Cellule 2.3 — Comptage des images par ensemble et par classe
count_rows = []
for name, df in splits.items():
    for label, n in df["label"].value_counts().reindex(class_names).items():
        count_rows.append(
            {"ensemble": name, "label": label, "classe": CLASS_DISPLAY.get(label, label), "nb_images": int(n)}
        )
split_counts_df = pd.DataFrame(count_rows)

totals = split_counts_df.groupby("ensemble")["nb_images"].sum().reindex(["train", "valid", "test"])
ratio_df = pd.DataFrame(
    {
        "nb_images": totals,
        "part": (totals / totals.sum()).round(4),
        "cible": pd.Series(SPLIT_RATIOS),
    }
)
ratio_df


In [ ]:
# Cellule 2.4 — Visualisation de la répartition (countplot empilé par ensemble)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(
    data=split_counts_df,
    x="ensemble",
    hue="classe",
    order=["train", "valid", "test"],
    palette="viridis",
    ax=ax1,
)
ax1.set_title("Nombre d'images par ensemble et par classe", weight="bold")
ax1.set_xlabel("Ensemble")
ax1.set_ylabel("Nombre d'images")
for container in ax1.containers:
    ax1.bar_label(container, fmt="%d", fontsize=8)

pivot = split_counts_df.pivot_table(index="classe", columns="ensemble", values="nb_images")
pivot = pivot[["train", "valid", "test"]]
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_norm.plot(kind="bar", stacked=True, ax=ax2,
                color=sns.color_palette("viridis", len(pivot_norm.columns)), width=0.7)
ax2.set_title("Proportions de classes par ensemble (%)", weight="bold")
ax2.set_xlabel("Classe")
ax2.set_ylabel("Part (%)")
ax2.legend(title="Ensemble", fontsize=8)

plt.tight_layout()
plt.show()


### 1.13 — Écriture du dataset nettoyé (RGB, redimensionné)

Les images sources sont hétérogènes (512×512, niveaux de gris, mode `RGBA`/`P`, 387
dimensions distinctes). Elles sont donc **normalisées** :

* conversion en **RGB** (les niveaux de gris sont dupliqués sur les 3 canaux) ;
* redimensionnement en **224×224** (taille attendue par GoogLeNet) ;
* ré-écriture en **JPEG qualité 95** dans `data/processed/classification/{train,valid,test}/<classe>/`.

Ce format est directement compatible avec `image_dataset_from_directory()` et accélère
considérablement la lecture des données pendant l'entraînement.

In [ ]:
# Cellule 2.5 — Materialisation du dataset nettoyé sur disque
from concurrent.futures import ThreadPoolExecutor

from PIL import Image

IMAGE_SIZE = 224
JPEG_QUALITY = 95


def write_processed_image(src_path: str, dst_path: str) -> tuple[str, bool]:
    # Convertit en RGB, redimensionne et ecrit l'image nettoyee.
    try:
        with Image.open(src_path) as im:
            im = im.convert("RGB")
            im = im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
            im.save(dst_path, format="JPEG", quality=JPEG_QUALITY)
        return dst_path, True
    except Exception:  # noqa: BLE001
        return dst_path, False


def build_processed_dataset(force: bool = False) -> pd.DataFrame:
    jobs = []
    for split, df in splits.items():
        for rec in df.to_dict("records"):
            dst_dir = PROCESSED_DIR / split / rec["label"]
            dst_dir.mkdir(parents=True, exist_ok=True)
            dst = dst_dir / Path(rec["file"]).with_suffix(".jpg").name
            jobs.append((rec, str(dst)))

    if not force:
        jobs = [(rec, dst) for rec, dst in jobs if not Path(dst).exists()]

    if jobs:
        with ThreadPoolExecutor(max_workers=8) as pool:
            results = list(pool.map(lambda j: write_processed_image(j[0]["path"], j[1]), jobs))
        failed = [dst for dst, ok in results if not ok]
        if failed:
            print(f"  {len(failed)} images non converties (ignorées)")

    manifest_rows = []
    for split, df in splits.items():
        for rec in df.to_dict("records"):
            dst = PROCESSED_DIR / split / rec["label"] / Path(rec["file"]).with_suffix(".jpg").name
            manifest_rows.append({"split": split, "label": rec["label"], "path": str(dst), "exists": dst.exists()})
    return pd.DataFrame(manifest_rows)


processed_manifest = build_processed_dataset(force=False)
print(f"Dataset nettoyé : {PROCESSED_DIR}")
for split in ["train", "valid", "test"]:
    sub = processed_manifest[processed_manifest["split"] == split]
    print(f"  {split:<6s} : {len(sub):>5d} images  ({sub['exists'].sum()} présentes sur disque)")


In [ ]:
# Cellule 2.6 — Comptage final des fichiers sur disque
for split in ["train", "valid", "test"]:
    for label in class_names:
        d = PROCESSED_DIR / split / label
        n = len(list(d.glob("*.jpg"))) if d.exists() else 0
        print(f"  {split:<6s} / {label:<12s} : {n:>5d}")


### 1.14 — Chargement par lots avec `image_dataset_from_directory()`

`tf.keras.utils.image_dataset_from_directory()` charge les images par **batches**,
applique automatiquement le **redimensionnement** (`image_size`) et la conversion en
float32. Le paramètre `shuffle=True` (entraînement) mélange les exemples à chaque époque.

In [ ]:
# Cellule 2.7 — Chargement des trois ensembles
import tensorflow as tf

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_dataset(split: str, shuffle: bool, seed: int | None = None) -> tf.data.Dataset:
    return tf.keras.utils.image_dataset_from_directory(
        str(PROCESSED_DIR / split),
        labels="inferred",
        label_mode="categorical",
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=seed,
        color_mode="rgb",
        interpolation="bilinear",
    )


train_ds = load_dataset("train", shuffle=True, seed=SEED)   # shuffle=True -> entraînement
valid_ds = load_dataset("valid", shuffle=False)              # pas de mélange -> évaluation stable
test_ds = load_dataset("test", shuffle=False)

train_ds = train_ds.cache().prefetch(AUTOTUNE)
valid_ds = valid_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

for name, ds in [("train", train_ds), ("valid", valid_ds), ("test", test_ds)]:
    n_batches = int(tf.data.experimental.cardinality(ds))
    print(f"{name:<6s} : {n_batches:>4d} lots de {BATCH_SIZE} images  "
          f"({n_batches * BATCH_SIZE} examples)")

xb, yb = next(iter(train_ds))
print(f"\nForme d'un lot : images {xb.shape} ({xb.dtype}) / labels {yb.shape} ({yb.dtype})")
print(f"Labels one-hot du premier exemple : {yb[0].numpy()}")


### 1.15 — Prétraitement Keras (redimensionnement, format, normalisation)

Le prétraitement est défini comme une **couche Keras** intégrée au modèle, ce qui le rend
autonome : le modèle entraîné acceptera directement des images `uint8` ou `float32`
en `[0, 255]` (cas de l'API FastAPI).

| Étape | Couche | Rôle |
|---|---|---|
| 1 | `Resizing(224, 224)` | redimensionnement (déjà fait à l'écriture, conservé par sécurité) |
| 2 | `Rescaling(1/255)` | conversion au format `float32` normalisé `[0, 1]` |
| 3 | `Normalization(mean, variance)` | centrage/réduction ImageNet, cohérent avec les poids pré-entraînés |

In [ ]:
# Cellule 2.8 — Pipeline de prétraitement Keras
from keras import layers

from brainscan.training.googlenet import IMAGENET_MEAN, IMAGENET_STD, build_input_pipeline

preprocess = build_input_pipeline(IMAGE_SIZE, augment=False)

raw = tf.cast(xb, tf.float32)
processed = preprocess(raw, training=False)

print("Pipeline de pretraitement (sans augmentation) :")
preprocess.summary(line_length=100)

print(f"\nEntree  : {raw.shape} {raw.dtype}  min={float(tf.reduce_min(raw)):.1f} max={float(tf.reduce_max(raw)):.1f}")
print(f"Sortie  : {processed.shape} {processed.dtype}")
print(f"moyenne par canal (apres normalisation) : {tf.reduce_mean(processed, axis=(0, 1, 2)).numpy().round(4)}")
print(f"ecart-type par canal                    : {tf.math.reduce_std(processed, axis=(0, 1, 2)).numpy().round(4)}")
print(f"\nStatistiques ImageNet utilisees : mean={IMAGENET_MEAN}, std={IMAGENET_STD}")


### 1.16 — Augmentation des données (ensemble d'entraînement uniquement)

Trois techniques sont appliquées, conformément au cahier des charges :

| Technique | Couche Keras 3 | Rôle |
|---|---|---|
| **Retournement** (flip) | `layers.RandomFlip("horizontal")` | symétrie gauche/droite |
| **Flou** (blur) | `layers.RandomGaussianBlur(kernel_size=3, sigma=(0.1, 1.5), factor=0.5)` | image floue / capteur bruité |
| **Bruit** (noise) | `layers.GaussianNoise(stddev=0.05)` | perturbation gaussienne des pixels |

**Pourquoi uniquement sur l'entraînement ?** Ces couches Keras n'appliquent la
transformation que lorsque `training=True`. Or Keras utilise `training=False` pendant
`model.evaluate()` et `model.predict()` : les ensembles de validation et de test restent
donc **strictement non augmentés**. La vérification ci-dessous le démontre empiriquement.

L'augmentation est intégrée au **modèle** (et non au dataset) afin que le modèle
entraîné reste autonome : il accepte des images `uint8` brutes et n'augmente jamais à
l'inférence.

In [ ]:
# Cellule 2.9 — Pipeline complet avec augmentation (identique à celui du modèle)
train_pipeline = build_input_pipeline(IMAGE_SIZE, augment=True)

print("Pipeline d'entraînement (prétraitement + augmentation) :")
train_pipeline.summary(line_length=100)


In [ ]:
# Cellule 2.10 — Démonstration : l'augmentation n'agit qu'en mode entraînement
sample = tf.cast(xb[:4], tf.float32)

out_train = train_pipeline(sample, training=True)   # transformations appliquées
out_eval = train_pipeline(sample, training=False)   # pipeline sans augmentation
reference = preprocess(sample, training=False)       # référence : prétraitement seul

print(f"Écart moyen |sortie - référence| en mode ENTRAÎNEMENT : {float(tf.reduce_mean(tf.abs(out_train - reference))):.5f}")
print(f"Écart moyen |sortie - référence| en mode INFERENCE  : {float(tf.reduce_mean(tf.abs(out_eval - reference))):.5f}")
print("\n-> En mode inference l'augmentation est désactivée : validation et test restent intacts.")


In [ ]:
# Cellule 2.11 — Visualisation des images augmentées (extrait du jeu d'entraînement)
n_show = 8
orig = np.array(xb[:n_show])
aug_norm = np.array(train_pipeline(tf.cast(xb[:n_show], tf.float32), training=True))
# dé-normalisation ImageNet pour l'affichage (le pipeline renvoie des valeurs normalisées)
aug = aug_norm * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)

fig, axes = plt.subplots(2, n_show, figsize=(1.9 * n_show, 4.6))
for i in range(n_show):
    axes[0, i].imshow(np.clip(orig[i], 0, 255).astype("uint8"))
    axes[0, i].set_title("original", fontsize=8)
    axes[1, i].imshow(np.clip(aug[i], 0, 1))
    axes[1, i].set_title("augmenté", fontsize=8)
    for r in (0, 1):
        axes[r, i].axis("off")
fig.suptitle("Augmentation des données (retournement, flou, bruit) — aperçu", weight="bold")
plt.tight_layout()
plt.show()


### 1.17 — Contrôles finaux

* proportions de classes par ensemble ;
* absence de doublon entre les trois ensembles ;
* répartition des effectifs.

In [ ]:
# Cellule 2.12 — Contrôles finaux du split
summary_rows = []
for split, df in splits.items():
    vc = df["label"].value_counts().reindex(class_names)
    summary_rows.append(
        {
            "ensemble": split,
            **{c: int(v) for c, v in vc.items()},
            "total": int(vc.sum()),
            "part": round(vc.sum() / len(manifest_df), 4),
        }
    )
summary_df = pd.DataFrame(summary_rows).set_index("ensemble").reindex(["train", "valid", "test"])
summary_df


In [ ]:
# Cellule 2.13 — Sauvegarde du split (référentiel pour les notebooks 3 et 4)
split_manifest = pd.concat(
    [df.assign(split=name) for name, df in splits.items()], ignore_index=True
)
split_manifest[["split", "label", "file", "path"]].to_csv(
    REPORTS_DIR / "classification_split.csv", index=False
)

summary_df.to_csv(REPORTS_DIR / "split_summary.csv")
print(f"Split sauvegardé : {REPORTS_DIR / 'classification_split.csv'}")
print(f"Résumé          : {REPORTS_DIR / 'split_summary.csv'}")
print("\nNotebook 2 terminé — passons à la construction du modèle GoogLeNet.")
